In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc samplomatic
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Ghid de început pentru Executor

<Accordion>
<AccordionItem title="Versiuni de pachete">

Codul de pe această pagină a fost dezvoltat folosind următoarele cerințe.
Recomandăm să folosești aceste versiuni sau mai noi.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
samplomatic~=0.18.0
```
</AccordionItem>
</Accordion>

Similar cu primitiva [Sampler](/guides/get-started-with-sampler), Executor eșantionează registrele de ieșire din execuțiile circuitelor cuantice, dar nu are nicio suprimare sau atenuare a erorilor integrată. În schimb, face parte din [modelul de execuție direcționată](/guides/directed-execution-model) care furnizează ingredientele pentru a capta intențiile de proiectare pe partea de client și transferă generarea costisitoare a variantelor de circuit pe partea de server. Executor urmează directivele furnizate în adnotările și opțiunile circuitului, generează și leagă valorile parametrilor, execută circuitele legate pe hardware și returnează rezultatele și metadatele execuției. Nu ia decizii implicite în locul tău și îți oferă control deplin și transparență.

> **Note:** Pachetul Qiskit nu are încă o clasă de bază pentru primitiva Executor.

## Înainte de a începe
Unele exemple de cod de pe această pagină folosesc `samplex`, care face parte din pachetul Samplomatic. Prin urmare, înainte de a rula acele blocuri de cod, trebuie să instalezi Samplomatic, așa cum se arată în blocul de cod următor. Pentru mai multe informații, consultă [documentația Samplomatic](https://qiskit.github.io/samplomatic).

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, Executor
from qiskit_ibm_runtime.quantum_program import QuantumProgram
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from samplomatic.transpiler import generate_boxing_pass_manager
from samplomatic import build

# Initialize the service and choose a backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

In [2]:
print(backend)

<IBMBackend('ibm_fez')>


### 2. Create and transpile a circuit

You need at least one circuit to use the Executor primitive.  It can optionally have parameters.

In [3]:
# Generate the circuit
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.h(1)
circuit.cz(0, 1)
circuit.h(1)

# Using `measure_all` automatically creates the necessary
# classical registers.
circuit.measure_all()

The circuit needs to be transformed to only use instructions supported by the QPU (referred to as *instruction set architecture (ISA)* circuits). Use the transpiler to do this.

In [4]:
# Transpile the circuit
preset_pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=0
)
isa_circuit = preset_pass_manager.run(circuit)

### 2. Creează și transpilează un circuit
Ai nevoie de cel puțin un circuit pentru a folosi primitiva Executor. Acesta poate avea, opțional, parametri.

In [5]:
# Initialize an empty program
program = QuantumProgram(shots=25)

# Append the circuit to the program
program.append_circuit_item(isa_circuit)

Circuitul trebuie transformat pentru a folosi doar instrucțiunile suportate de QPU (denumite circuite *cu set de instrucțiuni ISA*). Folosește transpilerul pentru aceasta.

In [6]:
# Generate a boxing pass manager to group gates
# and measurements into boxes and add
# a`Twirl` annotation.
boxes_pm = generate_boxing_pass_manager(
    # Add gate twirling
    enable_gates=True,
    # Add measurement twirling
    enable_measures=True,
)

boxed_circuit = boxes_pm.run(isa_circuit)
boxed_circuit.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/get-started-with-executor/extracted-outputs/245a4574-3ce9-4f77-98c8-af32cde8ac01-0.svg" alt="Output of the previous code cell" />

### 3. Inițializează un `QuantumProgram`
Inițializează un `QuantumProgram` cu volumul tău de lucru. Un `QuantumProgram` este alcătuit din `QuantumProgramItems`. De obicei, fiecare element constă dintr-un circuit, un set de valori de parametri și, eventual, un `samplex` pentru a randomiza conținutul circuitului. Pentru detalii complete, consultă [Intrările și ieșirile Executor](/guides/executor-input-output).

Celula următoare inițializează un `QuantumProgram` și specifică efectuarea a 25 de shot-uri. Apoi adaugă circuitul transpilat target.

In [7]:
# Build the template circuit and the samplex
template_circuit, samplex = build(boxed_circuit)

# Append the template circuit and samplex as a `samplex_item`
program.append_samplex_item(
    template_circuit,
    samplex=samplex,
    shape=(num_randomizations := 20,),
)

### 4. Opțional: Grupează porțile și măsurătorile în cutii adnotate
Gruparea instrucțiunilor în cutii și adnotarea acestora este modalitatea principală de a specifica intenția ta. În exemplul următor, folosim `generate_boxing_pass_manager` și parametrii săi de twirling pentru a grupa porțile cu doi qubiți și măsurătorile în cutii și a aplica adnotarea de twirling.

In [8]:
# Initialize an Executor with the default options
executor = Executor(mode=backend)

# Submit the job
job = executor.run(program)
job

<RuntimeJobV2('d8286580bvlc73d1vmsg', 'executor')>

In [9]:
# Retrieve the result
result = job.result()

### 6. Invocă Executor și obține rezultate
Rulează `QuantumProgram` pe un backend IBM&reg; folosind primitiva `Executor` cu opțiunile implicite. Consultă [Opțiunile Executor](/guides/executor-options) pentru a afla despre opțiunile disponibile.